In [ ]:


!pip install tensorflow==2.18.1 keras==3.10.0
import tensorflow as tf
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from pathlib import Path
!pip install ta
!pip install pandas_ta
import parser_data
import json
import birja2
from keras import layers as lay
import keras
import matplotlib.pyplot as plt
import Memory
import PPO_model2
import math




Исправить valeu_loss как приблежение к вектору которыы переддается в write_ или до read или после read для того чтобы сначала ии училась эти вектора делать а потом уже их в памти записывать и читать через  VAL_loss




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import datetime
from pathlib import Path
import birja2
import tensorflow as tf
tf.config.optimizer.set_jit(False)
#tf.debugging.enable_check_numerics()
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

#return {'window_data':column_stak,'action_one_hot':dirat,'scalars':scalar}
env = birja2.Trading_3()



In [ ]:
num_action=env.action_space
# {'scalars':3,'level':3} # 3 действия 3 уровня
gamma=0.99
lam=0.95

print(hasattr(env,'end_episode'))
#env.end_episode(2)

INFO:tensorflow:Enabled check-numerics callback in thread MainThread
start2: 2018-12-13 00:00:00+00:00 | index2: 2048 | start2(for index): 2018-12-13 00:00:00+00:00 end: 2025-12-24 00:00:00+00:00
start1: 2018-12-13 04:00:00+00:00 | index1: 277 | start1(for index): 2018-12-13 04:00:00+00:00 end: 2025-12-24 07:30:00+00:00
start3: 2018-12-03 00:00:00+00:00 | index3: 20 | start3(for index): 2018-12-03 00:00:00+00:00 end: 2025-12-15 00:00:00+00:00


True


In [ ]:
#@tf.py_function(Tout=[tf.float32,tf.bool])
def end_episode(reward):
            end_data=tf.constant([False,],tf.bool)
            if env.index1>=len(env.data_15)-1:
                end_data=tf.constant([True,],tf.bool)
            return tf.constant(reward,tf.float32),end_data

In [ ]:
#@tf.py_function(Tout=[tf.float32,tf.float32,tf.float32,tf.float32,tf.float32,tf.bool])
def step(action=0,cout=(0,0,0)):

    '''возвращение парамертров  step  в тензорах'''
    cout=tf.squeeze(cout)
    observation,reward__,done,trun,info=env.step(action=int(action),stop=float(cout[0]),take=float(cout[1]),open=float(cout[2]))
    #{'window_data':datas,"action_one_hot":dirat,"scalars":scalar,"global":global_level,'level':level,'un_pnl':un_pnl,'time_in_trade':time_in_t}
    #'15m':None,'4h':None,'1w':None
    with tf.device('/GPU:0'):
      observation_window={'15m':tf.reshape(tf.constant(observation['window_data']['15m'],tf.float32),(1,env.window_15,9)),
                          '4h':tf.reshape(tf.constant(observation['window_data']['4h'],tf.float32),(1,env.window_4+1,9)),
                          '1w':tf.reshape(tf.constant(observation['window_data']['1w'],tf.float32),(1,env.window_1+1,9))}

      observation={**observation_window,
                  'action_one_hot':tf.reshape(tf.constant(observation['action_one_hot'],tf.float32),(1,4,)),
                  'scalars':tf.reshape(tf.constant(observation['scalars'],tf.float32),(1,4,)),
                  'global':tf.reshape(tf.constant(observation['global'],tf.float32),(1,4,)),
                  'level':tf.reshape(tf.constant(observation['level'],tf.float32),(1,3,)),
                  'un_pnl':tf.reshape(tf.constant(observation['un_pnl'],tf.float32),(1,1,)),
                  'time_in_trade':tf.reshape(tf.constant(observation['time_in_trade'],tf.float32),(1,1,))
                  }
      reward__=tf.constant(reward__,tf.float32)
      done=done or trun
      done=tf.constant(float(done),tf.float32)
      trun=tf.constant(bool(done),tf.bool)
      #return observation[0],observation[1],observation[2],reward__,done,trun
      info1={}
      info1['1w']=tf.reshape(tf.constant(info['1w'],tf.bool),(1,1))
      info1['4h']=tf.reshape(tf.constant(info['4h'],tf.bool),(1,1))

    return observation,reward__,done,trun,info1

#@tf.py_function(Tout=[tf.float32,tf.float32,tf.float32])
def reset(e=True):
    if e:
        obs,info=env.reset()
        episode_=0
    else:
        with open(Path('/content/drive/MyDrive/Colab Notebooks/last_obs_8.json'),'r') as f:
            dat=json.load(f)
            episode_=dat.pop('episode')
        obs,info=env.get_obsevation_for_index(dat)
    with tf.device('/GPU:0'):
      observation_window={'15m':tf.reshape(tf.constant(obs['window_data']['15m'],tf.float32),(1,env.window_15,9)),
                          '4h':tf.reshape(tf.constant(obs['window_data']['4h'],tf.float32),(1,env.window_4+1,9)),
                          '1w':tf.reshape(tf.constant(obs['window_data']['1w'],tf.float32),(1,env.window_1+1,9))}

      obs={**observation_window,
                  'action_one_hot':tf.reshape(tf.constant(obs['action_one_hot'],tf.float32),(1,4,)),
                  'scalars':tf.reshape(tf.constant(obs['scalars'],tf.float32),(1,4,)),
                  'global':tf.reshape(tf.constant(obs['global'],tf.float32),(1,4,)),
                  'level':tf.reshape(tf.constant(obs['level'],tf.float32),(1,3,)),
                  'un_pnl':tf.reshape(tf.constant(obs['un_pnl'],tf.float32),(1,1,)),
                  'time_in_trade':tf.reshape(tf.constant(obs['time_in_trade'],tf.float32),(1,1,))
                  }
      #return obs[0],obs[1],obs[2]
      tf.print(obs.keys())
      info1={}
      info1['1w']=tf.reshape(tf.constant(info['1w'],tf.bool),(1,1))
      info1['4h']=tf.reshape(tf.constant(info['4h'],tf.bool),(1,1))
      info1['epis']=episode_
    return obs,info1

#jit_compile=True
@tf.function()
def get_discounted_sum(x,gamma):
    x=tf.cast(x[::-1],tf.float32)
    gamma=tf.constant(gamma,tf.float32)
    y=tf.TensorArray(tf.float32,0,dynamic_size=True)
    y0=tf.constant(0.0)
    y0_shape=y0.shape
    for i in tf.range(tf.shape(x)[0]):
        y0=x[i]+y0*gamma
        y0=tf.reshape(y0,y0_shape)
        y=y.write(i,y0)

    return y.stack()[::-1]
#jit_compile=True
@tf.function()
def log_probability_get(logits,action):
    #log=tf.math.log_softmax(logits)
    log=tf.nn.log_softmax(logits)
    return tf.reduce_sum(tf.one_hot(tf.cast(action,tf.int32),num_action['scalars'])*log,-1)
@tf.function()
def log_kl_get(logits,action,logits_old):
    #log=tf.math.log_softmax(logits)
    logit_new=tf.nn.log_softmax(logits,-1)
    #new=tf.reduce_sum(tf.one_hot(tf.cast(action,tf.int32),num_action['scalars'])*logit_new,-1)
    logit_old=tf.nn.log_softmax(logits_old,-1)
    #old=tf.reduce_sum(tf.one_hot(tf.cast(action,tf.int32),num_action['scalars'])*logit_old,-1)
    return logit_old-logit_new




<>:38: SyntaxWarning: invalid escape sequence '\c'
<>:38: SyntaxWarning: invalid escape sequence '\c'
C:\Users\Comp\AppData\Local\Temp\ipykernel_11872\397043549.py:38: SyntaxWarning: invalid escape sequence '\c'
  with open(Path('AI\cripto_bot\models\env_last_obs (1).json'),'r') as f:


In [ ]:
# создание крикитов
def critic_create() -> keras.Model:
    tf1=lay.Input((env.window_15,9),name='15m')
    tf2=lay.Input((env.window_4+1,9),name='4h')
    tf3=lay.Input((env.window_1+1,9),name='1w')
    act_hot=lay.Input((4,),name='action_one_hot')
    sclr=lay.Input((4,),name='scalars')
    glob=lay.Input((4,),name='global')
    lvl=lay.Input((3,),name='level')
    un_pnl=lay.Input((1,),name='un_pnl')
    t_in_tr=lay.Input((1,),name='time_in_trade')

    # внутрение слои
    conv_1=lay.Conv1D(32,5,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(tf1)
    conv_2=lay.Conv1D(64,7,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(tf2)
    conv_3=lay.Conv1D(128,5,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(tf3)

    conv_1=lay.Conv1D(48,3,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(conv_1)
    conv_2=lay.Conv1D(64,5,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(conv_2)
    transform1=PPO_model2.Transformer_block(num_head=4,key_dim=12,d_ff=96,num=1,value_dim=16,l=1,num_experts=4,top_k=2)(conv_1)
    transform2=PPO_model2.Transformer_block(num_head=4,key_dim=16,d_ff=128,num=1,value_dim=16,l=1,num_experts=4,top_k=2)(conv_2)
    transform3=PPO_model2.Transformer_block(num_head=8,key_dim=16,d_ff=256,num=1,value_dim=16,l=1,num_experts=4,top_k=2)(conv_3)

    gl_max1=lay.GlobalAveragePooling1D()(transform1)
    gl_max2=lay.GlobalAveragePooling1D()(transform2)
    gl_max3=lay.GlobalAveragePooling1D()(transform3)

    # read1=Memory.ReadMemory(mem_15m,48,8)(gl_max1,can_r=can_r15,vector_h=vector_h15)
    # read2=Memory.ReadMemory(mem_4h,64,16)(gl_max2,can_r=can_r4,vector_h=vector_h4)
    # read3=Memory.ReadMemory(mem_1w,128,16)(gl_max3,can_r=can_r1,vector_h=vector_h1)

    sclr1=lay.Dense(8,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(sclr)
    act_hot1=lay.Dense(8,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(act_hot)
    glob1=lay.Dense(8,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(glob)
    lvl1=lay.Dense(8,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(lvl)
    un_pnl1=lay.Dense(4,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(un_pnl)
    t_in_tr1=lay.Dense(4,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(t_in_tr)

    context=lay.Concatenate()([gl_max1,gl_max2,gl_max3,sclr1,act_hot1,glob1,lvl1,un_pnl1,t_in_tr1])

    value=lay.Dense(1,kernel_regularizer=keras.regularizers.l2(1e-4))(context)
    critic=keras.Model(inputs={'15m':tf1,'4h':tf2,'1w':tf3,'action_one_hot':act_hot,'scalars':sclr,'global':glob,'level':lvl,'un_pnl':un_pnl,'time_in_trade':t_in_tr},
                    outputs=value)
    return critic

In [ ]:
# инициализация слоев и среды
observations_data,info=reset(False)
epis=info['epis']
mem_15m = Memory.Memory(128,48,48, name_='15', name='memory_15')
mem_4h  = Memory.Memory(128,64,64, name_='4', name='memory_4')
mem_1w  = Memory.Memory(64,128,128, name_='1', name='memory_1')

output_cout_l=PPO_model2.Output_cout(2)
output_cout_o=PPO_model2.Output_cout(1)

# входные слои
tf1=lay.Input((env.window_15,9),name='15m')
tf2=lay.Input((env.window_4+1,9),name='4h')
tf3=lay.Input((env.window_1+1,9),name='1w')
act_hot=lay.Input((4,),name='action_one_hot')
sclr=lay.Input((4,),name='scalars')
glob=lay.Input((4,),name='global')
lvl=lay.Input((3,),name='level')
un_pnl=lay.Input((1,),name='un_pnl')
t_in_tr=lay.Input((1,),name='time_in_trade')
can_w15=lay.Input((1,),dtype=tf.bool,name='can_write15')
can_w4=lay.Input((1,),dtype=tf.bool,name='can_write4')
can_w1=lay.Input((1,),dtype=tf.bool,name='can_write1')
can_r15=lay.Input((1,),dtype=tf.bool,name='can_read15')
can_r4=lay.Input((1,),dtype=tf.bool,name='can_read4')
can_r1=lay.Input((1,),dtype=tf.bool,name='can_read1')
vector_h15=lay.Input((48,),dtype=tf.float32,name='vector_r15')
vector_h4=lay.Input((64,),dtype=tf.float32,name='vector_r4')
vector_h1=lay.Input((128,),dtype=tf.float32,name='vector_r1')

# внутрение слои
conv_1=lay.Conv1D(32,5,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(tf1)
conv_2=lay.Conv1D(64,7,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(tf2)
conv_3=lay.Conv1D(128,5,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(tf3)

conv_1=lay.Conv1D(48,3,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(conv_1)
conv_2=lay.Conv1D(64,5,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(conv_2)
#conv_3=lay.Conv1D(128,3,activation='gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(conv_3)

transform1=PPO_model2.Transformer_block(num_head=4,key_dim=12,d_ff=96,num=1,value_dim=16,l=6,num_experts=8,top_k=2)(conv_1)
transform2=PPO_model2.Transformer_block(num_head=4,key_dim=16,d_ff=128,num=1,value_dim=16,l=4,num_experts=8,top_k=2)(conv_2)
transform3=PPO_model2.Transformer_block(num_head=8,key_dim=16,d_ff=256,num=1,value_dim=16,l=2,num_experts=8,top_k=2)(conv_3)

transform1=PPO_model2.Transformer_block(num_head=4,key_dim=12,d_ff=96,num=2,value_dim=16,l=6,num_experts=8,top_k=2)(transform1)
transform2=PPO_model2.Transformer_block(num_head=4,key_dim=16,d_ff=128,num=2,value_dim=16,l=4,num_experts=8,top_k=2)(transform2)
transform3=PPO_model2.Transformer_block(num_head=8,key_dim=16,d_ff=256,num=2,value_dim=16,l=2,num_experts=8,top_k=2)(transform3)

transform1=PPO_model2.Transformer_block(num_head=4,key_dim=12,d_ff=96,num=3,value_dim=16,l=6,num_experts=8,top_k=2)(transform1)
transform2=PPO_model2.Transformer_block(num_head=4,key_dim=16,d_ff=128,num=3,value_dim=16,l=4,num_experts=8,top_k=2)(transform2)

transform1=PPO_model2.Transformer_block(num_head=4,key_dim=12,d_ff=96,num=4,value_dim=16,l=6,num_experts=8,top_k=2)(transform1)
transform2=PPO_model2.Transformer_block(num_head=4,key_dim=16,d_ff=128,num=4,value_dim=16,l=4,num_experts=8,top_k=2)(transform2)

transform1=PPO_model2.Transformer_block(num_head=4,key_dim=12,d_ff=96,num=5,value_dim=16,l=6,num_experts=8,top_k=2)(transform1)

transform1=PPO_model2.Transformer_block(num_head=4,key_dim=12,d_ff=96,num=6,value_dim=16,l=6,num_experts=8,top_k=2)(transform1)

gl_max1=lay.GlobalAveragePooling1D()(transform1)
gl_max2=lay.GlobalAveragePooling1D()(transform2)
gl_max3=lay.GlobalAveragePooling1D()(transform3)

# блок чтения и записи памяти
read1=Memory.ReadMemory(mem_15m,48,8)(gl_max1,can_r=can_r15,vector_h=vector_h15)
read2=Memory.ReadMemory(mem_4h,64,16)(gl_max2,can_r=can_r4,vector_h=vector_h4)
read3=Memory.ReadMemory(mem_1w,128,16)(gl_max3,can_r=can_r1,vector_h=vector_h1)

k_new1,v_new1,p1=Memory.WriteMemory(mem_15m,tf_name='15m',o_f=0.4,n_f=0.05,a_f=1,s_max=0.65)(gl_max1,can_write=can_w15)
k_new2,v_new2,p2=Memory.WriteMemory(mem_4h,tf_name='4h',o_f=0.6,n_f=0.03,a_f=1,s_max=0.65)(gl_max2,can_write=can_w4)
k_new3,v_new3,p3=Memory.WriteMemory(mem_1w,tf_name='1w',o_f=0.7,n_f=0.02,a_f=1,s_max=0.65)(gl_max3,can_write=can_w1)


read=lay.Concatenate()([read1,read2,read3])
context=lay.Concatenate()([gl_max1,gl_max2,gl_max3])

context=lay.Dense(128,'gelu',kernel_regularizer=keras.regularizers.l2(1e-3))(context)
read=lay.Dense(128,'gelu',kernel_regularizer=keras.regularizers.l2(1e-3))(read)
context=lay.LayerNormalization()(context)
read=lay.LayerNormalization()(read)


sclr1=lay.Dense(16,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(sclr)
act_hot1=lay.Dense(16,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(act_hot)
glob1=lay.Dense(16,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(glob)
lvl1=lay.Dense(16,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(lvl)
un_pnl1=lay.Dense(8,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(un_pnl)
t_in_tr1=lay.Dense(8,'gelu',kernel_regularizer=keras.regularizers.l2(1e-4))(t_in_tr)


context=lay.Concatenate()([context,read,sclr1,act_hot1,glob1,lvl1,un_pnl1,t_in_tr1])
context1=lay.Dropout(0.1)(context)
context=lay.Dense(128,'gelu',kernel_regularizer=keras.regularizers.l2(1e-3))(context1)
context=lay.LayerNormalization()(context)

# выходные слои
scal_act=lay.Dense(num_action['scalars'])(context)
#mean_l=lay.Dense(2)(context)
#std_l=lay.Dense(2,activation='softplus')(context)
mean_l,std_l=output_cout_l(context)
mean_l=20*keras.activations.tanh(mean_l/20)

mean_o,std_o=output_cout_o(context)
#mean_o=lay.Dense(1)(context)
#std_o=lay.Dense(1,activation='softplus')(context)
mean_o=10*keras.activations.tanh(mean_o/10)

predV=lay.Dense(1)(lay.concatenate([v_new1,v_new2,v_new3]))
actor=keras.Model(inputs={'15m':tf1,'4h':tf2,'1w':tf3,'action_one_hot':act_hot,'scalars':sclr,'global':glob,'level':lvl,'un_pnl':un_pnl,'time_in_trade':t_in_tr,'can_write15':can_w15,'can_write4':can_w4,'can_write1':can_w1,
                          'can_read15':can_r15,'can_read4':can_r4,'can_read1':can_r1,'vector_r15':vector_h15,'vector_r4':vector_h4,'vector_r1':vector_h1},
                  outputs={'scal_act':scal_act,'mean_l':mean_l,'mean_o':mean_o,
                           'std_l':std_l,'std_o':std_o,
                           'k1':k_new1,'v1':v_new1,'p1':p1,'gl_max1':gl_max1,'vector_r15':read1,
                           'k2':k_new2,'v2':v_new2,'p2':p2,'gl_max2':gl_max2,'vector_r4':read2,
                           'k3':k_new3,'v3':v_new3,'p3':p3,'gl_max3':gl_max3,'vector_r1':read3,'predv':predV})


# critic disk
critic=critic_create()
critic_cout=critic_create()
critic.summary()
if False:
    keras.saving.load_weights(actor,Path("/content/drive/MyDrive/Colab Notebooks/models/actor_last_008.weights.h5"))
    keras.saving.load_weights(critic,Path("/content/drive/MyDrive/Colab Notebooks/models/critic_last_008.weights.h5"))
    for i in actor.weights:
        print(i)

#keras.utils.plot_model(actor,'model_0.0.png',True,True,True)
#keras.utils.plot_model(critic,'model_0.1.png',True,True,True)



['15m',
 '4h',
 '1w',
 'action_one_hot',
 'scalars',
 'global',
 'level',
 'un_pnl',
 'time_in_trade']


<>:139: SyntaxWarning: invalid escape sequence '\P'
<>:140: SyntaxWarning: invalid escape sequence '\P'
<>:139: SyntaxWarning: invalid escape sequence '\P'
<>:140: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Comp\AppData\Local\Temp\ipykernel_11872\1039599689.py:139: SyntaxWarning: invalid escape sequence '\P'
  keras.saving.load_weights(actor,Path("D:\Python1/AI\cripto_bot\models\models_actor_last_001.weights.h5"))
C:\Users\Comp\AppData\Local\Temp\ipykernel_11872\1039599689.py:140: SyntaxWarning: invalid escape sequence '\P'
  keras.saving.load_weights(critic,Path("D:\Python1/AI/cripto_bot/models/models_critic_last_001.weights.h5"))


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ 15m (InputLayer)    │ (None, 150, 9)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 4h (InputLayer)     │ (None, 100, 9)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 146, 32)   │      1,472 │ 15m[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 94, 64)    │      4,096 │ 4h[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 1w (InputLayer)     │ (None, 20, 9)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_18 (Conv1D)  │ (None, 144, 48)   │      4,656 │ conv1d_15[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_19 (Conv1D)  │ (None, 90, 64)    │     20,544 │ conv1d_16[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 16, 128)   │      5,888 │ 1w[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_… │ (None, 144, 48)   │     70,580 │ conv1d_18[0][0]   │
│ (Transformer_block) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_… │ (None, 90, 64)    │     92,152 │ conv1d_19[0][0]   │
│ (Transformer_block) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_… │ (None, 16, 128)   │    331,008 │ conv1d_17[0][0]   │
│ (Transformer_block) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 48)        │          0 │ transformer_bloc… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ transformer_bloc… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ transformer_bloc… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ can_read15          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vector_r15          │ (None, 48)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ can_read4           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vector_r4           │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ can_read1           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 577,101 (2.31 MB)

 Trainable params: 501,981 (1.91 MB)

 Non-trainable params: 75,120 (407.08 KB)

<Variable path=conv1d_15/kernel, shape=(5, 9, 32), dtype=float32, value=[[[ 0.02658417  0.09398834  0.02516872 ...  0.16791046  0.1310953
   -0.10268865]
  [-0.10663132 -0.17350392 -0.04744407 ... -0.004494   -0.05143182
   -0.10576337]
  [ 0.03149347 -0.00421087 -0.0321826  ... -0.1634292  -0.07760417
   -0.07151271]
  ...
  [-0.1496091  -0.04788262  0.00774576 ... -0.1705583   0.15188633
   -0.03310252]
  [-0.13357118 -0.16897202 -0.15345298 ... -0.09667955  0.00196845
   -0.04508403]
  [-0.02055585 -0.07276074  0.02267857 ...  0.00626296 -0.01068352
    0.03872485]]

 [[ 0.12491467 -0.10037246 -0.12154248 ...  0.14014748  0.05437297
   -0.00929148]
  [-0.05425251 -0.11770623  0.07459084 ...  0.01732995 -0.00868238
    0.1226351 ]
  [ 0.04093656  0.11928274  0.14010005 ...  0.12904587 -0.12325507
   -0.08733889]
  ...
  [ 0.06483986 -0.01458899  0.0211853  ...  0.03245091  0.12780552
   -0.10336111]
  [ 0.11440788  0.10361768 -0.14625454 ... -0.06434728  0.04968616
    0.05408142]
  

SyntaxError: invalid decimal literal (1656813835.py, line 63)

In [ ]:
#optimizer_policy=keras.optimizers.Adam(2e-6)
optimizer_policy=keras.optimizers.Adam(1e-5)
read_f=tf.constant(False,tf.bool)
i_=tf.constant(0)
i_1=tf.constant(671)
i_2=tf.constant(671)

eps_log=tf.ones((32,),tf.float32)*1e-6
eps_kl=tf.ones((32,3),tf.float32)*1e-6

game=tf.Variable(tf.zeros([],tf.float32))
reward=tf.constant([0.,],tf.float32)
end_data=tf.constant(False,tf.bool)
critic_m=tf.constant(False,tf.bool)
read_m=tf.constant(True,tf.bool)
lnpi=tf.math.log(math.pi*2)
eps_l=tf.ones((1,2),tf.float32)*1e-5
eps_o=tf.ones((1,1),tf.float32)*1e-5
scr_max=tf.ones((1,3),tf.float32)*0.5
scr_min=tf.ones((1,3),tf.float32)*1e-3

#scr_min=tf.ones((1,3),tf.float32)*1e-3
mask_key=1.0-tf.eye(32)

@tf.function()
def train_policy(logits,observations,last_obs,rewards,dones,log_prababilitys,actions,vector_15,vector_4,vector_1,cout,std_old,mean_old,entropy,los,kl,pol_l,v_l,imp_l,k_l,epoh_c,rat_min,rat_max,rat_mean,rat_min_c,rat_max_c,rat_mean_c,kl_c,kl_d,ent_d,ent_c,epoh_key,clip_frac,ra_min,ra_max,ra_mean,std_n_min,std_n_max,std_n_mean,mask_action):
    grads=[tf.zeros_like(i) for i in actor.trainable_variables]

    #inicial params
    batch_size=32
    values=tf.zeros((0,1),tf.float32)
    values_cout=tf.zeros((0,1),tf.float32)

    # create values and values_cout
    cond=lambda i,val,val1: i<672
    def val_i(i,values,values_cout):
        values=tf.concat([values,critic({'15m':observations['15m'][i:i+32],'4h':observations['4h'][i:i+32],'1w':observations['1w'][i:i+32],'level':observations['level'][i:i+32],'action_one_hot':observations['action_one_hot'][i:i+32],'un_pnl':observations['un_pnl'][i:i+32],'time_in_trade':observations['time_in_trade'][i:i+32],'global':observations['global'][i:i+32],'scalars':observations['scalars'][i:i+32]})])
        values_cout=tf.concat([values_cout,critic_cout({'15m':observations['15m'][i:i+32],'4h':observations['4h'][i:i+32],'1w':observations['1w'][i:i+32],'level':observations['level'][i:i+32],'action_one_hot':observations['action_one_hot'][i:i+32],'un_pnl':observations['un_pnl'][i:i+32],'time_in_trade':observations['time_in_trade'][i:i+32],'global':observations['global'][i:i+32],'scalars':observations['scalars'][i:i+32]})])  # 672
        return i+batch_size,values,values_cout
    _,values,values_cout=tf.while_loop(cond,
                                       val_i,
                                       (i_,values,values_cout),
                                       parallel_iterations=1, swap_memory=False)
    # создание последнего value
    last_value=critic(last_obs)
    last_value_cout=critic_cout(last_obs)

    # инициализация параметров для подсчета gae
    gae=tf.zeros_like(last_value)
    advantages=tf.TensorArray(tf.float32,672,dynamic_size=False, clear_after_read=True)
    advantages_cout=tf.TensorArray(tf.float32,672,dynamic_size=False, clear_after_read=True)
    gae_shape=tf.shape(gae)

    def gae_i(ste,last_value,gae,advantages,rewards,values,dones):
        delta=rewards[ste]+gamma*(1-dones[ste])*last_value-values[ste]
        gae=tf.reshape(delta+gamma*lam*(1-dones[ste])*gae,gae_shape)
        advantages=advantages.write(ste,gae)
        last_value=tf.reshape(values[ste],gae_shape)
        return ste-1,last_value,gae,advantages
    
    # подсчет gae 
    cond=lambda i,_1,_2,_3,_4,_5,_6: i>=0
    _,last_value,gae,advantages=tf.while_loop(cond,gae_i,
                                   (i_1,last_value,gae,advantages,rewards,values,dones),swap_memory=False,parallel_iterations=1)
    gae=tf.zeros_like(last_value)
    _,last_value_cout,gae,advantages_cout=tf.while_loop(cond,gae_i,
                                   (i_2,last_value_cout,gae,advantages_cout,rewards,values_cout,dones),swap_memory=False,parallel_iterations=1)
    
    advantages=advantages.stack()
    advantages_cout = advantages_cout.stack()

    returns=advantages+values
    returns_cout=advantages_cout+ values_cout

    advantages=(advantages-tf.reduce_mean(advantages))/(tf.math.reduce_std(advantages)+1e-8)

    action_cout=tf.reduce_sum(mask_action)
    advantages_cout=mask_action*advantages_cout

    adv_mean=tf.reduce_sum(advantages_cout)/(action_cout+1e-8)
    adv_var=tf.reduce_sum((((advantages_cout-adv_mean)*mask_action)**2))/(action_cout+1e-8)
    adv_std=tf.sqrt(adv_var)+1e-8
    advantages_cout=((advantages_cout - adv_mean)/adv_std)*mask_action
    
    
    cond = lambda i, g, tl, te,io,ik,ioj,ioh,rm,rmax,r_mean,r_min_c,r_max_c,r_mean_c,ent_d,ent_c,cl_f,ra,re,rt,sd_n_min,sd_n_max,sd_n_mean: i < 672
    def body(i,grads,los,entropy,pol_l,v_l,imp_l,k_l,rat_min,rat_max,rat_mean,rat_min_c,rat_max_c,rat_mean_c,ent_d,ent_c,clip_frac,ra_min,ra_max,ra_mean,std_n_min,std_n_max,std_n_mean):
        batch_size_cur = tf.shape(actions[i:i+32])[0]  # 32, но может быть меньше последний батч? У вас всегда 32, т.к. 672/32=21
        flags = tf.fill((batch_size_cur, 1), read_f)
        with tf.GradientTape() as tape:
            resolt=actor({'15m':observations['15m'][i:i+32],'4h':observations['4h'][i:i+32],'1w':observations['1w'][i:i+32],'level':observations['level'][i:i+32],'action_one_hot':observations['action_one_hot'][i:i+32],'un_pnl':observations['un_pnl'][i:i+32],'time_in_trade':observations['time_in_trade'][i:i+32],'global':observations['global'][i:i+32],'scalars':observations['scalars'][i:i+32],'can_write15':flags,'can_write4':flags,'can_write1':flags,'can_read15':flags,'can_read4':flags,'can_read1':flags,
                    'vector_r15':vector_15[i:i+32],'vector_r4':vector_4[i:i+32],'vector_r1':vector_1[i:i+32]},training=True)
            log=log_probability_get(resolt['scal_act'],actions[i:i+32])

            std=tf.concat([resolt['std_l'],resolt['std_o']],-1)

            u=tf.concat([resolt['mean_l'],resolt['mean_o']],-1)
            #std= scr_min+(scr_max-scr_min)*std
            #std= tf.clip_by_value(std,-20.0,2.0)
            #std=tf.exp(std)
            std=0.01+0.3*std
            std=tf.clip_by_value(std,0.01,1.5)
            std_n_max=tf.maximum(tf.reduce_max(std),std_n_max)
            std_n_min=tf.minimum(tf.reduce_min(std),std_n_min)
            std_n_mean+=tf.reduce_mean(std)
            varianse=tf.clip_by_value(std**2,1e-6,100.0)
            log_cout_n=-0.5*tf.reduce_sum(((cout[i:i+32]-u)**2)/varianse+2*tf.math.log(std)+lnpi,-1)
            new_log=log+log_cout_n
            
            ratio=tf.exp(new_log-log_prababilitys[i:i+32])
            ra_min=tf.minimum(tf.reduce_min(ratio),ra_min)
            ra_max=tf.maximum(tf.reduce_max(ratio),ra_max)
            ra_mean+=tf.reduce_mean(ratio)

            rat_max=tf.maximum( tf.reduce_max(log),rat_max)
            rat_min=tf.minimum(tf.reduce_min(log),rat_min)
            rat_mean+=tf.reduce_mean(log)

            rat_max_c=tf.maximum( tf.reduce_max(log_cout_n),rat_max_c)
            rat_min_c=tf.minimum(tf.reduce_min(log_cout_n),rat_min_c)
            rat_mean_c+=tf.reduce_mean(log_cout_n)
            # min_advantage=tf.where(advantages[i:i+32]>0,(1+0.2)*advantages[i:i+32],(1-0.2)*advantages[i:i+32])
            # policy_loss=-tf.reduce_mean(tf.minimum(ratio*advantages[i:i+32],min_advantage))
            min_advantage_d= tf.clip_by_value(ratio,1-0.2,1+0.2)
            #min_advantage_c= tf.clip_by_value(ratio_c,1-0.2,1+0.2)
            loss=-tf.reduce_mean(tf.minimum(ratio*advantages[i:i+32],min_advantage_d*advantages[i:i+32]))
            #loss_c=-tf.reduce_mean(tf.minimum(ratio_c*advantages[i:i+32],min_advantage_c*advantages[i:i+32]))

            # policy_loss=loss_d+1.0*loss_c
            # pol_l+=loss_d+1.0*loss_c
            policy_loss=loss
            pol_l+=loss
            #pol_l+=-tf.reduce_mean(tf.minimum(ratio*advantages[i:i+32],min_advantage*advantages[i:i+32]))
            entropy_disk = tf.reduce_mean(
            -tf.reduce_sum(tf.nn.softmax(resolt['scal_act']) * tf.nn.log_softmax(resolt['scal_act']), axis=-1)
            )
            ent_d+=entropy_disk

            val_loss=tf.reduce_mean(tf.square(resolt['predv']-tf.reshape(returns[i:i+32],(-1,1))))
            imp_loss=tf.reduce_mean(tf.square(tf.concat([resolt['p1'],resolt['p2'],resolt['p3']],-1)-tf.clip_by_value(tf.expand_dims(tf.abs(advantages[i:i+32]),-1)/3,0.0,1.0)))

            key_norm=tf.nn.l2_normalize(tf.concat([resolt['k1'],resolt['k2'],resolt['k3']],-1),-1)
            sim_k=tf.matmul(key_norm,key_norm,transpose_b=True)
            key_loss_start=tf.reduce_mean(tf.square(sim_k*mask_key))

            ret=tf.reshape(returns[i:i+32],(-1,))
            ret=(ret-tf.reduce_mean(ret)) / (tf.math.reduce_std(ret)+1e-6)
            target=tf.abs(ret[:,None]-ret[None,:])*mask_key

            v_norm=tf.nn.l2_normalize(tf.concat([resolt['gl_max1'],resolt['gl_max2'],resolt['gl_max3']],-1),-1)
            sim_v=tf.matmul(v_norm,v_norm,transpose_b=True)
            key_loss_midl=tf.reduce_mean((1.0+0.2*target)*tf.square((sim_k-tf.stop_gradient(sim_v))*mask_key))
            
            #key_loss_end=tf.reduce_mean(tf.square(sim_k-target))
            key_loss=epoh_key*key_loss_start+ (1-epoh_key)*key_loss_midl

            #key_loss=tf.reduce_mean(1-tf.matmul(tf.nn.l2_normalize(tf.concat([resolt['gl_max1'],resolt['gl_max2'],resolt['gl_max3']],-1),-1),tf.concat([resolt['k1'],resolt['k2'],resolt['k3']],-1),transpose_b=True))
            #policy_loss+=0.9*val_loss+0.5*imp_loss+0.2*key_loss
            policy_loss+=0.1*val_loss+0.1*imp_loss+0.1*key_loss

            entropy_cout=tf.reduce_mean(0.5*tf.reduce_sum((tf.math.log(2*math.pi) +2*tf.math.log(std)+1),-1))
            ent_c+=entropy_cout

            policy_loss -= epoh_c * entropy_disk+epoh_c*entropy_cout
            pol_l-=epoh_c * entropy_disk+epoh_c*entropy_cout
        grad=tape.gradient(policy_loss,actor.trainable_variables)
        los+=policy_loss
        grads=[grad_pr+gr for grad_pr,gr in zip(grads,grad)]
        entropy+= epoh_c * entropy_disk+epoh_c*entropy_cout
        
        clip_frac += tf.reduce_mean(
        tf.cast(tf.abs(ratio- 1.0) > 0.2,tf.float32))

        return i+batch_size,grads,los,entropy,pol_l,v_l+val_loss*0.1,imp_l+imp_loss*0.1,k_l+key_loss*0.1,rat_min,rat_max,rat_mean,rat_min_c,rat_max_c,rat_mean_c,ent_d,ent_c,clip_frac,ra_min,ra_max,ra_mean,std_n_min,std_n_max,std_n_mean
    _, grads, los, entropy,pol_l,v_l,imp_l,k_l,rat_min,rat_max,rat_mean,rat_min_c,rat_max_c,rat_mean_c,ent_d,ent_c,clip_frac,ra_min,ra_max,ra_mean,std_n_min,std_n_max,std_n_mean = tf.while_loop(
        cond, body, (i_, grads, los, entropy,pol_l,v_l,imp_l,k_l,rat_min,rat_max,rat_mean,rat_min_c,rat_max_c,rat_mean_c,ent_d,ent_c,clip_frac,ra_min,ra_max,ra_mean,std_n_min,std_n_max,std_n_mean),
        parallel_iterations=1, swap_memory=False
    )
    grads=[o/21.0 for o in grads]
    grads, _ = tf.clip_by_global_norm(grads, 6.0)
    optimizer_policy.apply_gradients(zip(grads,actor.trainable_variables))
 
    cond_kl = lambda i, kl_acc,kl_c,kl_d: i < 672

# def log_kl_get(logits,action,logits_old):
#     #log=tf.math.log_softmax(logits)
#     logit_new=tf.nn.log_softmax(logits,-1)
#     #new=tf.reduce_sum(tf.one_hot(tf.cast(action,tf.int32),num_action['scalars'])*logit_new,-1)
#     logit_old=tf.nn.log_softmax(logits_old,-1)
#     #old=tf.reduce_sum(tf.one_hot(tf.cast(action,tf.int32),num_action['scalars'])*logit_old,-1)
#     return logit_old-logit_new
    def body1(i,kl,kl_c,kl_d):
        batch_size_cur = tf.shape(actions[i:i+32])[0]  # 32, но может быть меньше последний батч? У вас всегда 32, т.к. 672/32=21
        flags = tf.fill((batch_size_cur, 1), read_f)
        new_resolt=actor({'15m':observations['15m'][i:i+32],'4h':observations['4h'][i:i+32],'1w':observations['1w'][i:i+32],'level':observations['level'][i:i+32],'action_one_hot':observations['action_one_hot'][i:i+32],'un_pnl':observations['un_pnl'][i:i+32],'time_in_trade':observations['time_in_trade'][i:i+32],'global':observations['global'][i:i+32],'scalars':observations['scalars'][i:i+32],'can_write15':flags,'can_write4':flags,'can_write1':flags,'can_read15':flags,'can_read4':flags,'can_read1':flags,
                    'vector_r15':vector_15[i:i+32],'vector_r4':vector_4[i:i+32],'vector_r1':vector_1[i:i+32]},training=True)
        
        logits_new=tf.nn.softmax(new_resolt['scal_act'],-1)
        kl_disc=tf.reduce_sum(logits_new*(tf.nn.log_softmax(new_resolt['scal_act'],-1)-tf.nn.log_softmax(logits[i:i+32])),-1)


        std=tf.concat([new_resolt['std_l'],new_resolt['std_o']],-1)
        #std= scr_min+(scr_max-scr_min)*std
        #std= tf.clip_by_value(std,-20.0,2.0)
        #std=tf.exp(std)
        std=0.01+0.3*std
        std=tf.clip_by_value(std,0.01,1.5)

        u=tf.concat([new_resolt['mean_l'],new_resolt['mean_o']],-1)
        log_kl_c=tf.math.log(std/(std_old[i:i+32]+eps_kl))

        kl_cout=log_kl_c+(tf.square(std_old[i:i+32])+tf.square(mean_old[i:i+32]-u))/(2*tf.square(std)+eps_kl)-0.5
        kl_cout=tf.reduce_sum(kl_cout,-1)
        kl_c+=tf.reduce_mean(kl_cout)
        kl_d+=tf.reduce_mean(kl_disc)
        kl+=tf.reduce_mean(kl_cout+kl_disc)
        return i+batch_size,kl,kl_c,kl_d
    _, kl,kl_c,kl_d = tf.while_loop(
        cond_kl, body1, (i_, kl,kl_c,kl_d),
        parallel_iterations=1, swap_memory=False
    )
    #kl=tf.reduce_sum(kl)

    return kl/21.0,los/21.0,entropy/21.0,pol_l/21.0,v_l/21.0,imp_l/21.0,k_l/21.0,rat_min,rat_max,rat_mean/21.0,rat_min_c,rat_max_c,rat_mean_c/21.0,kl_c/21.0,kl_d/21.0,ent_d/21.0,ent_c/21.0,clip_frac/21.0,ra_min,ra_max,ra_mean/21.0,std_n_min,std_n_max,std_n_mean/21.0


#optimizer_critic=keras.optimizers.Adam(1e-5)
optimizer_critic=keras.optimizers.Adam(3e-5)
@tf.function
def train_critic(observations,rewards,dones,mask_cout,vector_15,vector_4,vector_1,los,last_obs):
    #inicial params
    batch_size=32
    
    grads=[tf.zeros_like(i) for i in critic.trainable_variables]

    cond = lambda i,g,gh: i < 672
    def body(i,grads,los):     # для can_read*
        flags_critic = tf.fill((batch_size, 1), critic_m)   # critic_m – скаляр
        flags_read   = tf.fill((batch_size, 1), read_f)     # read_f – скаляр
        with tf.GradientTape() as tape:
            critic_loss=tf.reduce_mean((returns[i:i+32]-critic({'15m':observations['15m'][i:i+32],'4h':observations['4h'][i:i+32],'1w':observations['1w'][i:i+32],'level':observations['level'][i:i+32],'action_one_hot':observations['action_one_hot'][i:i+32],'un_pnl':observations['un_pnl'][i:i+32],'time_in_trade':observations['time_in_trade'][i:i+32],'global':observations['global'][i:i+32],'scalars':observations['scalars'][i:i+32],'can_write15':flags_critic,'can_write4':flags_critic,'can_write1':flags_critic,'can_read15':flags_read,'can_read4':flags_read,'can_read1':flags_read,
                    'vector_r15':vector_15[i:i+32],'vector_r4':vector_4[i:i+32],'vector_r1':vector_1[i:i+32]}))**2)
        grad=tape.gradient(critic_loss,critic.trainable_variables)
        grads=[grad_pr+gr for grad_pr,gr in zip(grads,grad)]
        return i+batch_size,grads,los+critic_loss

    _,grads,los  = tf.while_loop(
        cond, body, (i_, grads,los),
        parallel_iterations=1, swap_memory=False
    )
    grads=[o/21.0 for o in grads]
    grads, _ = tf.clip_by_global_norm(grads, 6.0)
    optimizer_critic.apply_gradients(zip(grads,critic.trainable_variables))


    return los/21.0


1. использовать маску для дейсвия которые ничего не делают
2. считать все отдельно до loss и непосредственно сами loss слаживать с маской
3. перепроверить все формулы и поменять kl_cout так как не так считается
4. разделение critic


https://share.google/aimode/Gl2t90mlGFCtGD7qE

In [ ]:
batch_dim=1
DATA_SHAPE=(150,9)
ACTIONS_SHAPE=(3,)
SCAL_SHAPE=(6,)
#{'window_data':column_stak,'action_one_hot':dirat,'scalars':scalar}
#jit_compile=True

@tf.function(jit_compile=True)
def actor_(obs):
    return actor(obs,training=False)
@tf.function(jit_compile=True)
def critic_(obs):
    return critic(obs,training=False)


import tensorflow as tf
import math
with tf.device('/GPU:0'):
  # Предполагается, что у вас уже определены:
  # actor, critic, log_probability_get, read_f, critic_m, lnpi

  lnpi = tf.math.log(2 * math.pi)

  # --- Создание фиктивных данных для построения графа ---
  num_steps = 672
  batch_size = 32

  observations_dummy = {
      '15m': tf.zeros((num_steps, 150, 9), tf.float32),
      '4h': tf.zeros((num_steps, 100, 9), tf.float32),
      '1w': tf.zeros((num_steps, 20, 9), tf.float32),
      'action_one_hot': tf.zeros((num_steps, 4), tf.float32),
      'scalars': tf.zeros((num_steps, 4), tf.float32),
      'global': tf.zeros((num_steps, 4), tf.float32),
      'level': tf.zeros((num_steps, 3), tf.float32),
      'un_pnl': tf.zeros((num_steps, 1), tf.float32),
      'time_in_trade': tf.zeros((num_steps, 1), tf.float32),
      # Флаги должны иметь размер (num_steps, 1), чтобы срез работал
      'can_write15': tf.zeros((num_steps,1,), tf.bool),
      'can_write4': tf.zeros((num_steps,1,), tf.bool),
      'can_write1': tf.zeros((num_steps,1,), tf.bool),
      'can_read15': tf.ones((num_steps,1,), tf.bool),
      'can_read4': tf.ones((num_steps,1,), tf.bool),
      'can_read1': tf.ones((num_steps,1,), tf.bool),
  }

  log_probs_disk_dummy = tf.zeros((num_steps,), tf.float32)
  log_probs_cout_dummy = tf.zeros((num_steps,), tf.float32)
  actions_disk_dummy = tf.zeros((num_steps,), tf.int32)
  advantages_dummy = tf.zeros((num_steps,), tf.float32)
  vector_15_dummy = tf.zeros((num_steps, 48), tf.float32)
  vector_4_dummy = tf.zeros((num_steps, 64), tf.float32)
  vector_1_dummy = tf.zeros((num_steps, 128), tf.float32)
  actions_cout_dummy = tf.zeros((num_steps, 3), tf.float32)
  returns_dummy = tf.zeros((num_steps,), tf.float32)
  logits = tf.zeros((num_steps, 3), tf.float32)
  std_old=tf.zeros((num_steps, 3), tf.float32)
  mean_old=tf.zeros((num_steps, 3), tf.float32)
  # Дополнительные аргументы (если они всё ещё в сигнатуре)
  entropy_dummy = tf.constant([0.0,])
  los_dummy = tf.constant([0.0,])
  kl_dummy = tf.constant([0.0,])

  # --- Вызов функций для построения графа ---
  # train_policy

  train_policy(
      logits,
      observations_dummy,
      log_probs_disk_dummy,
      actions_disk_dummy,
      advantages_dummy,
      vector_15_dummy,
      vector_4_dummy,
      vector_1_dummy,
      actions_cout_dummy,
      returns_dummy,
      std_old,
      mean_old,
      entropy_dummy,
      los_dummy,
      kl_dummy,tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([1.0,],tf.float32),tf.constant(100.0),tf.constant(0.0),tf.constant(0.0),tf.constant(100.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0))

  # train_critic
  train_critic(observations_dummy, returns_dummy, vector_15_dummy, vector_4_dummy, vector_1_dummy,tf.constant([0.0,]))


  stat_trajectory=tf.Variable(tf.zeros([batch_dim,],tf.int32))
  game=tf.Variable(tf.zeros([],tf.float32))
  reward=tf.constant([0.,],tf.float32)
  end_data=tf.constant(False,tf.bool)
  critic_m=tf.constant(False,tf.bool)
  read_m=tf.constant(True,tf.bool)
  lnpi=tf.math.log(math.pi*2)





  dummy_obs = {
      '15m': tf.zeros((1,150,9), tf.float32),
      '4h': tf.zeros((1,100,9), tf.float32),
      '1w': tf.zeros((1,20,9), tf.float32),
      'action_one_hot': tf.zeros((1,4), tf.float32),
      'scalars': tf.zeros((1,4), tf.float32),
      'global': tf.zeros((1,4), tf.float32),
      'level': tf.zeros((1,3), tf.float32),
      'un_pnl': tf.zeros((1,1), tf.float32),
      'time_in_trade': tf.zeros((1,1), tf.float32),
      'can_write15': tf.zeros((1,1), tf.bool),
      'can_write4': tf.zeros((1,1), tf.bool),
      'can_write1': tf.zeros((1,1), tf.bool),
      'can_read15': tf.ones((1,1), tf.bool),
      'can_read4': tf.ones((1,1), tf.bool),
      'can_read1': tf.ones((1,1), tf.bool),
      'vector_r15': tf.zeros((1,48), tf.float32),
      'vector_r4': tf.zeros((1,64), tf.float32),
      'vector_r1': tf.zeros((1,128), tf.float32),
  }



  #initial
  actor_(dummy_obs)
  critic_(dummy_obs)

In [ ]:

#@tf.function
#-------------------------------------------------
#env.unwrapped.current_step=34322-672
#-------------------------------------------------
#-------------------------------------------------
std_min1=tf.ones((1,2),tf.float32)*2e-2
std_min2=tf.ones((1,1),tf.float32)*2e-2

import datetime
def start_episode(step_per_episode,observation_data,info,gamma=0.99,lam=0.95,rand_ac=False):
    global end_data
    if end_data:
        observation_data,info=reset()
        end_data=tf.constant([False,],tf.bool)
    #'15m''4h'   '1w''action_one_hot''level''global''un_pnl''time_in_trade''scalars'
    observation_15=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    observation_4=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    observation_1=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    observation_globs=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    observation_scals=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    observation_act_one=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    observation_lvl=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    observation_tint=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)
    observation_unpnl=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)

    log_probs_cout=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    actions_cout=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    vector_15=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    vector_4=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    vector_1=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    std_old=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)
    mean_old=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True,)

    rewards=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)
    logits=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)
    dones=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)
    actions_disk=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)
    values=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)
    log_probs_disk=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)

    mask_action=tf.TensorArray(tf.bool,step_per_episode,dynamic_size=False, clear_after_read=True)

    game.assign(tf.zeros([],tf.float32))
    logical_bool=tf.constant(False,tf.bool)

    for ste in range(step_per_episode):
        #tf.print('step:',ste)
        #now=datetime.datetime.now()
        if not logical_bool:
            obs_d1,obs_d2,obs_d3,obs_action,obs_level,obs_glob,obs_unpl,obs_time,obs_scal=(observation_data['15m'],observation_data['4h'],
                                                                    observation_data['1w'],observation_data['action_one_hot'],observation_data['level'],observation_data['global'],observation_data['un_pnl'],observation_data['time_in_trade'],observation_data['scalars'])
            # (tf.expand_dims(tf.squeeze(observation_data['15m']),0),
            # tf.expand_dims(tf.squeeze(observation_data['4h']),0),
            # tf.expand_dims(tf.squeeze(observation_data['1w']),0),
            # tf.expand_dims(tf.squeeze(observation_data['action_one_hot']),0),
            # tf.expand_dims(tf.squeeze(observation_data['level']),0),
            # tf.expand_dims(tf.squeeze(observation_data['global']),0),
            # tf.expand_dims(tf.squeeze(observation_data['un_pnl']),0),
            # tf.expand_dims(tf.squeeze(observation_data['time_in_trade']),0),
            # tf.expand_dims(tf.squeeze(observation_data['scalars']),0),)
            # obs_d1,obs_d2,obs_d3,obs_action,obs_level,obs_glob,obs_unpl,obs_time,obs_scal=(tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['15m']),(150,9)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['4h']),(100,9)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['1w']),(20,9)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['action_one_hot']),(4,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['level']),(3,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['global']),(4,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['un_pnl']),(1,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['time_in_trade']),(1,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['scalars']),(4,)),0),tf.float32),)
            can1=tf.cast(tf.reshape(tf.squeeze(info['1w']),(1,1)),tf.bool)
            can4=tf.cast(tf.reshape(tf.squeeze(info['4h']),(1,1)),tf.bool)
            can15=tf.cast(tf.reshape(tf.constant(True),(1,1)),tf.bool)


            # {'scal_act':scal_act,'mean_l':mean_l,'mean_o':mean_o,
            #                'std_l':std_l,'std_o':std_o,
            #                'k1':k_new1,'v1':v_new1,'p1':p1,'gl_max1':gl_max1,
            #                'k2':k_new2,'v2':v_new2,'p2':p2,'gl_max2':gl_max2,
            #                'k3':k_new3,'v3':v_new3,'p3':p3,'gl_max3':gl_max3,}
            read_m_reshaped = tf.reshape(read_m, (1,1))
            obs={'15m':obs_d1,'4h':obs_d2,'1w':obs_d3,'action_one_hot':obs_action,'scalars':obs_scal,'global':obs_glob,'level':obs_level,'un_pnl':obs_unpl,'time_in_trade':obs_time,
                 'can_write15':can15,'can_write4':can4,'can_write1':can1,'can_read15':read_m_reshaped,'can_read4':read_m_reshaped,'can_read1':read_m_reshaped,
                 'vector_r15':tf.zeros((1,48,),tf.float32),'vector_r4':tf.zeros((1,64,),tf.float32),'vector_r1':tf.zeros((1,128,),tf.float32)}
            #tf.print('colelate pred resolt:',(datetime.datetime.now()-now).total_seconds())#-------------------------------------

            resolt=actor_(obs)
            #tf.print('colelate resolt:',(datetime.datetime.now()-now).total_seconds())#-------------------------------------

            logit=tf.cast(tf.reshape(resolt['scal_act'],(3,)),tf.float32)
            #std_l= tf.clip_by_value(resolt['std_l'],-20.0,2.0)
            #std_l=tf.exp(std_l)
            #std_o= tf.clip_by_value(resolt['std_o'],-20.0,2.0)
            #std_o=tf.exp(std_o)
            std_l=resolt['std_l']
            std_o=resolt['std_o']
            a_st=tf.reshape(std_l,(2,))*tf.random.normal((2,)) + resolt['mean_l']
            a_o=tf.reshape(std_o,(1,))*tf.random.normal((1,)) + resolt['mean_o']
            cout=tf.concat([a_st,a_o],-1)
            std=tf.concat([std_l,std_o],-1)
            varianse=tf.clip_by_value(std**2,1e-6,100.0)
            u=tf.concat([resolt['mean_l'],resolt['mean_o']],-1)


            if not rand_ac:
                action=tf.cast(tf.random.categorical(tf.expand_dims(logit,0),1),tf.float32)
            else:
                action=tf.cast(tf.random.uniform((),0,3,tf.int32),tf.float32)
            

            observation_data_new,reward,done,logical_bool,info=step(action,cout=cout)
            reward=tf.cast(tf.squeeze(reward),tf.float32)
            if logical_bool:
                reward_,end_data=end_episode(reward)
               
            read_m_reshaped = tf.reshape(read_m, (1,1))
            critic_m_reshaped = tf.reshape(critic_m, (1,1))

            observation_15 = observation_15.write(ste,obs_d1)
            observation_4 = observation_4.write(ste,obs_d2)
            observation_1 = observation_1.write(ste,obs_d3)
            observation_act_one = observation_act_one.write(ste,obs_action)
            observation_lvl = observation_lvl.write(ste,obs_level)
            observation_unpnl = observation_unpnl.write(ste,obs_unpl)
            observation_tint = observation_tint.write(ste,obs_time,)
            observation_globs = observation_globs.write(ste,obs_glob)
            observation_scals = observation_scals.write(ste,obs_scal)
        

            log_cout_n=-0.5*tf.reduce_sum(((cout-u)**2)/varianse+2*tf.math.log(std)+lnpi,-1)
            
            mask_action.write(ste,action!=0)
            rewards=rewards.write(ste,reward)
            logits=logits.write(ste,logit)
            log_probs_disk=log_probs_disk.write(ste,log_probability_get(logit,action)+log_cout_n)
            dones=dones.write(ste,done)
            
            actions_disk=actions_disk.write(ste,action)
            actions_cout=actions_cout.write(ste,cout)
            std_old=std_old.write(ste,std)
            mean_old=mean_old.write(ste,u)
            #log_probs_cout=log_probs_cout.write(ste,tf.reduce_sum(-0.5*((cout-u)/std)**2-tf.math.log(std)-lnpi,-1))
            log_probs_cout=log_probs_cout.write(ste,cout-u)
            vector_15=vector_15.write(ste,resolt['vector_r15'])
            vector_4=vector_4.write(ste,resolt['vector_r4'])
            vector_1=vector_1.write(ste,resolt['vector_r1'])

            game.assign_add(tf.squeeze(reward))
            #tf.print('write resolt:',(datetime.datetime.now()-now).total_seconds())#-------------------------------------

            #observation_data=observation_new['window_data']
            #observation_action=observation_new['action_one_hot']
            #observation_scal=observation_new['scalars']
            observation_data=observation_data_new
            obs_d1,obs_d2,obs_d3,obs_action,obs_level,obs_glob,obs_unpl,obs_time,obs_scal=(observation_data['15m'],observation_data['4h'],
                                                                    observation_data['1w'],observation_data['action_one_hot'],observation_data['level'],observation_data['global'],observation_data['un_pnl'],observation_data['time_in_trade'],observation_data['scalars'])
            
        else:
            tf.print(f'logit=True:{ste}')
            # obs_d1,obs_d2,obs_d3,obs_action,obs_level,obs_glob,obs_unpl,obs_time,obs_scal=(tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['15m']),(150,9)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['4h']),(100,9)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['1w']),(20,9)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['action_one_hot']),(4,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['level']),(3,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['global']),(4,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['un_pnl']),(1,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['time_in_trade']),(1,)),0),tf.float32),
            # tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data['scalars']),(4,)),0),tf.float32),
            # )
            obs_d1,obs_d2,obs_d3,obs_action,obs_level,obs_glob,obs_unpl,obs_time,obs_scal=(observation_data['15m'],observation_data['4h'],
                                                                    observation_data['1w'],observation_data['action_one_hot'],observation_data['level'],observation_data['global'],observation_data['un_pnl'],observation_data['time_in_trade'],observation_data['scalars'])

            observation_15 = observation_15.write(ste,obs_d1)
            observation_4 = observation_4.write(ste,obs_d2)
            observation_1 = observation_1.write(ste,obs_d3)
            observation_act_one = observation_act_one.write(ste,obs_action)
            observation_lvl = observation_lvl.write(ste,obs_level)
            observation_unpnl = observation_unpnl.write(ste,obs_unpl)
            observation_tint = observation_tint.write(ste,obs_time)
            observation_globs = observation_globs.write(ste,obs_glob)
            observation_scals = observation_scals.write(ste,obs_scal)

            mask_action.write(ste,tf.constant(False,tf.bool))
            rewards=rewards.write(ste,tf.constant(0.0,tf.float32))
            logits=logits.write(ste,tf.constant([0.,0.,0.],tf.float32))
            log_probs_disk=log_probs_disk.write(ste,tf.constant(0.0,tf.float32))
            dones=dones.write(ste,tf.constant(1.,tf.float32))
            
            end_data=tf.constant([False,],tf.bool)
            actions_disk=actions_disk.write(ste,tf.constant(0.0,tf.float32))
            actions_cout=actions_cout.write(ste,tf.constant([0.,0.,0.],tf.float32))
            std_old=std_old.write(ste,tf.constant([0.,0.,0.],tf.float32))
            mean_old=mean_old.write(ste,tf.constant([0.,0.,0.],tf.float32))
            log_probs_cout=log_probs_cout.write(ste,tf.constant(0.0,tf.float32))

            vector_15= vector_15.write(ste,tf.zeros((48,),tf.float32))
            vector_4= vector_4.write(ste,tf.zeros((64,),tf.float32))
            vector_1= vector_1.write(ste,tf.zeros((128,),tf.float32))

            #tf.print('write val:',(datetime.datetime.now()-now).total_seconds())#-------------------------------------

        #tf.print('time end:',(datetime.datetime.now()-now).total_seconds())
            #game.assign_add(tf.squeeze(reward))

        #tf.print('time end:',(datetime.datetime.now()-now).total_seconds())#-------------------------------------

    # tf.print("end_ste:",tf.shape(observation_datas))


    logical_bool=tf.constant(False,tf.bool)

    observation_15 = observation_15.stack()
    observation_4 = observation_4.stack()
    observation_1 = observation_1.stack()
    observation_act_one = observation_act_one.stack()
    observation_lvl = observation_lvl.stack()
    observation_unpnl = observation_unpnl.stack()
    observation_tint = observation_tint.stack()
    observation_globs = observation_globs.stack()
    observation_scals = observation_scals.stack()

    mask_action=mask_action.stack()
    log_probs_cout=log_probs_cout.stack()
    actions_cout=actions_cout.stack()
    std_old=std_old.stack()
    mean_old=mean_old.stack()
    rewards=rewards.stack()
    logits=logits.stack()
    dones=dones.stack()
    actions_disk=actions_disk.stack()
    values=critic({'15m':observation_15,'4h':observation_4,'1w':observation_1,'action_one_hot':observation_act_one,'scalars':observation_scals,'global':observation_globs,'level':observation_lvl,'un_pnl':observation_unpnl,'time_in_trade':observation_tint})
    values_cout=critic_cout({'15m':observation_15,'4h':observation_4,'1w':observation_1,'action_one_hot':observation_act_one,'scalars':observation_scals,'global':observation_globs,'level':observation_lvl,'un_pnl':observation_unpnl,'time_in_trade':observation_tint})
    log_probs_disk=log_probs_disk.stack()

    vector_15=vector_15.stack()
    vector_4=vector_4.stack()
    vector_1=vector_1.stack()

    
    can1=tf.reshape(info['1w'],(1,1,))
    can4=tf.reshape(info['4h'],(1,1))
    can15=tf.ones((1,1), dtype=tf.bool)

    obs_d1,obs_d2,obs_d3,obs_action,obs_level,obs_glob,obs_unpl,obs_time,obs_scal=(observation_data['15m'],observation_data['4h'],
                                                                    observation_data['1w'],observation_data['action_one_hot'],observation_data['level'],observation_data['global'],observation_data['un_pnl'],observation_data['time_in_trade'],observation_data['scalars'])
    read_m_reshaped = tf.reshape(read_m, (1,1))
    critic_m_reshaped = tf.reshape(critic_m, (1,1))

    last_value=critic_({'15m':obs_d1,'4h':obs_d2,'1w':obs_d3,'action_one_hot':obs_action,'scalars':obs_scal,'global':obs_glob,'level':obs_level,'un_pnl':obs_unpl,'time_in_trade':obs_time,'can_write15':critic_m_reshaped,'can_write4':critic_m_reshaped,'can_write1':critic_m_reshaped,'can_read15':read_m_reshaped,'can_read4':read_m_reshaped,'can_read1':read_m_reshaped,
                 'vector_r15':tf.zeros((1,48,),tf.float32),'vector_r4':tf.zeros((1,64,),tf.float32),'vector_r1':tf.zeros((1,128,),tf.float32)})
    last_value=tf.squeeze(last_value)

    last_observation={'15m':obs_d1,'4h':obs_d2,'1w':obs_d3,'action_one_hot':obs_action,'scalars':obs_scal,'global':obs_glob,'level':obs_level,'un_pnl':obs_unpl,'time_in_trade':obs_time}

    gae=tf.zeros_like(last_value)
    gae_shape=tf.shape(gae)
    advantages=tf.TensorArray(tf.float32,step_per_episode,dynamic_size=False, clear_after_read=True)
    for ste in tf.range(tf.shape(rewards)[0]-1,-1,-1):
        delta=rewards[ste]+gamma*(1-dones[ste])*last_value-values[ste]
        gae=tf.reshape(delta+gamma*lam*(1-dones[ste])*gae,gae_shape)
        advantages=advantages.write(ste,gae)

        last_value=tf.reshape(values[ste],gae_shape)
    advantages=advantages.stack()

    returns=advantages+values
    #tf.print("end_ste:",tf.shape(observation_datas))


    advantages=(advantages-tf.reduce_mean(advantages))/(tf.math.reduce_std(advantages)+1e-8)
    #________________END DATA!!!_______________
    #observation_15: [672 1 150 9] | observation_4: [672 1 100 9] | observation_1: [672 1 20 9] | action_one_hot: [672 1 4] | level: [672 1 3] | unpnl: [672 1 1] | time_in_trade: [672 1 1] | globals: [672 1 4] | scalars: [672 1 4]
    # | log_probs_cout: [672 1] | actions_cout: [672 1 3] | rewards: [672] | logits: [672 3] | dones: [672] | actions_disk: [672 1 1] | values: [672] | log_probs_disk: [672 1 1] | vector_15: [672 1 48] | vector_4: [672 1 64] | vector_1: [672 1 128] | context1: [672 1 48] | context2: [672 1 64] | context3: [672 1 128] | keys_new1: [672 1 32] | keys_new2: [672 1 48] | keys_new3: [672 1 64] | val_new1: [672 1 48] | val_new2: [672 1 64] | val_new3: [672 1 128] | imp_new1: [672 1 1] | imp_new2: [672 1 1] | imp_new3: [672 1 1]
    observations={'15m':tf.squeeze(observation_15),'4h':tf.squeeze(observation_4),'1w':tf.squeeze(observation_1),'level':tf.squeeze(observation_lvl),'action_one_hot':tf.squeeze(observation_act_one),'un_pnl':tf.reshape((observation_unpnl),(-1,1)),'time_in_trade':tf.squeeze(observation_tint),'global':tf.squeeze(observation_globs),'scalars':tf.squeeze(observation_scals)}
    rewards=tf.reshape(rewards,(-1,))
    dones=tf.reshape(dones,(-1,))
    mask_action=tf.reshape(mask_action,(-1,))
    log_probs_disk=tf.reshape(log_probs_disk,(-1,))
    returns=tf.reshape(returns,(-1,))
    actions_disk=tf.reshape(actions_disk,(-1,))
    #log_probs_cout=tf.reshape(log_probs_cout,(-1,))
    values=tf.reshape(values,(-1,))
    actions_cout=tf.reshape(actions_cout,(-1,3))
    mean_old=tf.reshape(mean_old,(-1,3))
    std_old=tf.reshape(std_old,(-1,3))


    advantages=tf.reshape(advantages,(-1,))
    tf.print(f"-----------------reward:{tf.reduce_sum(rewards)}| | last value:{last_value}| value_mean:{tf.reduce_mean(values)}---------------------------------------------------------------------------------------------------")

    #return observations,rewards,dones,log_probs_disk,returns,actions,values,advantages,logits,observation_data,observation_action,observation_scal
    return observations,rewards,dones,log_probs_disk,returns,actions_disk,values,advantages,logits,actions_cout,std_old,mean_old,observation_data,tf.squeeze(vector_15),tf.squeeze(vector_4),tf.squeeze(vector_1),info,log_probs_cout,mask_action,last_observation


In [ ]:
reward_min=-10000

# было бы у меня еще 3-4гб доп на ноутбуке смог бы кешировать все состояния в среде и ускорить до 0.01-0.001c
import cProfile
import pstats
import io
with tf.device('/GPU:0'):
  # Начинаем профилирование
  # profiler = cProfile.Profile()
  # profiler.enable()
  try:
      for episode in range(epis,50000):
          rand_ac=False
          if episode<3:
              rand_ac=True

          (observations,rewards,dones,log_probs_disk,returns,actions_disk,values,advantages,logits,actions_cout,std_old,mean_old,observation_data,vector_15,vector_4,vector_1,info,log_probs_cout,mask_action)=start_episode(96*7,observations_data,info,rand_ac)

          '''print(f'obs_w:{tf.shape(observations["window_data"])}')
          print(f'obs_a:{tf.shape(observations["action_one_hot"])}')
          print(f'obs_s:{tf.shape(observations["scalars"])}')

          print(f'rew:{tf.shape(rewards)}')
          print(f'act:{tf.shape(actions)}')
          print(f'val:{tf.shape(values)}')
          print(f'adv:{tf.shape(advantages)}')
          print(f'log_pr:{tf.shape(log_probabilitis)}')
          print(f'ret:{tf.shape(returns)}')
          print(f'don:{tf.shape(dones)}')'''
          tf.print(f'std_old min:{tf.reduce_min(std_old)}| max:{tf.reduce_max(std_old)}| mean:{tf.reduce_mean(std_old)}| mean_old min:{tf.reduce_min(mean_old)}| max:{tf.reduce_max(mean_old)}| mean:{tf.reduce_mean(mean_old)}| ation_cout min:{tf.reduce_min(actions_cout)}| acctions_cout max:{tf.reduce_max(actions_cout)}| mean:{tf.reduce_mean(actions_cout)}')
          tf.print(f'std_old first 15: {std_old[:15]} | mean_old fist 15: {mean_old[:15]}')
          tf.print(f'actions_cout first 15: {actions_cout[:15]}')
          if episode>=0:
              old_w=[tf.identity(v) for v in actor.trainable_variables]
              for i in range(15):
                      tf.print('обучение:',i,'| balanse:',env.balance,end='| ')
                      
                          #observations,log_prababilitys,log_cout,actions,advantages,logits,vector_15,vector_4,vector_1,cout
                      kl,loss,ent,pol_l,v_l,imp_l,k_l,rat_min,rat_max,rat_mean,rat_min_c,rat_max_c,rat_mean_c,kl_c,kl_d,ent_d,ent_c,clip_f,ra_min,ra_max,ra_mean,std_n_min,std_n_max,std_n_mean=train_policy(logits,observations,log_probs_disk,actions_disk,advantages,vector_15,vector_4,vector_1,actions_cout,returns,std_old,mean_old,tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([0.0,]),tf.constant([max(0.0001,0.01 * (0.995 ** episode)),],tf.float32),tf.constant(100.0),tf.constant(-100.0),tf.constant(0.0),tf.constant(100.0),tf.constant(-100.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(0.0),tf.constant(max(0.2,1.0-episode/180.0)),tf.constant(0.0),tf.constant(300.0),tf.constant(-300.0),tf.constant(0.0),tf.constant(100.0),tf.constant(-100.0),tf.constant(0.0),mask_action)
                      tf.print(f'log_min_d:{rat_min}| log_max_d:{rat_max}| log_mean_d:{rat_mean}| log_min_c:{rat_min_c} | log_max_c:{rat_max_c}| log_mean_c:{rat_mean_c}| kl_cout:{kl_c}| kl_disk:{kl_d} | ent_d:{ent_d}| ent_c:{ent_c}| c-m_min:{tf.reduce_min(log_probs_cout)}| c-m_max:{tf.reduce_max(log_probs_cout)}| c-m_mean:{tf.reduce_mean(log_probs_cout)}|rat_min: {ra_min}| rat_max:{ra_max}| rat_mean:{rat_mean}| std_n_min:{std_n_min}| std_n_max:{std_n_max} |std_n_mean:{std_n_mean}|  clip_fac:{clip_f}')
                      if kl>1.5*0.1:
                          break
              delta=[]
              for w_old, w_new in zip(old_w, actor.trainable_variables):
                delta.append( tf.reduce_mean(tf.abs(w_new - w_old)))
              delta_act = tf.reduce_mean(delta)

          else:
              loss=0.
              ent=0
              kl=0
              pol_l,v_l,imp_l,k_l=0,0,0,0
          old_w=[tf.identity(v) for v in critic.trainable_variables]
          for i in range(15):
              loss_critic=train_critic(observations,returns,vector_15,vector_4,vector_1,tf.constant([0.0,]))
          delta=[]
          for w_old, w_new in zip(old_w, critic.trainable_variables):
                delta.append( tf.reduce_mean(tf.abs(w_new - w_old)))
          delta_crit = tf.reduce_mean(delta)


          '''
          if episode%40==0:
              for w in actor.trainable_variables:
                  w.assign_add(tf.random.normal(tf.shape(w), stddev=0.05))'''
          if tf.reduce_sum(tf.reshape(rewards, (-1)), 0) > reward_min:
            actor.save_weights(r"/content/drive/MyDrive/Colab Notebooks/models/actor_305.weights.h5")
            critic.save_weights(r"/content/drive/MyDrive/Colab Notebooks/models/critic_305.weights.h5")
            obs=env.get_index(env._get_observation())
            with open('/content/drive/MyDrive/Colab Notebooks/best_obs_305.json','w') as f:
                obs['episode']=episode
                json.dump(obs,f,indent=2)
          if episode % 1 == 0:

            actor.save_weights(r"/content/drive/MyDrive/Colab Notebooks/models/actor_last_305.weights.h5")
            critic.save_weights(r"/content/drive/MyDrive/Colab Notebooks/models/critic_last_305.weights.h5")
            obs=env.get_index(env._get_observation())
            with open('/content/drive/MyDrive/Colab Notebooks/last_obs_305.json','w') as f:
                obs['episode']=episode
                json.dump(obs,f,indent=2)
            reward_min=tf.reduce_sum(tf.reshape(rewards,(-1,)),0)
          #print('action_disk:',actions_disk[20:40],'| action_cout:',actions_cout[20:40])


          print(f'Epochs:{episode}|rewards mean:{tf.reduce_mean(tf.reshape(rewards,(-1,)))}| rewards:{reward_min}|loss:{loss}|kl:{kl}|ent:{ent}| pol_l:{pol_l} | v_l:{v_l} | imp_l:{imp_l} | k_l:{k_l}| delta_act:{delta_act} | delta_crit:{delta_crit}| loss_critic:{loss_critic}')

          print('-------------------------')
  except KeyboardInterrupt as s:
      # profiler.disable()

      # # Выводим результаты
      # s = io.StringIO()
      # stats = pstats.Stats(profiler, stream=s).sort_stats('cumulative')
      # stats.print_stats(50)  # показать 20 самых медленных функций
      # print(s.getvalue())
      print('k')

In [ ]:
1 это не critic в обычном понимании это для того чтобы обучать в памяти value_net то есть чтобы модель сохраняла инфу помагающую определять награду то есть была полесной и еще мне в идеале сделать чтобы градиент далше чем этоот слой dense не проходил именно по этому пути от dense val_net дальше нельзя 





In [ ]:
1 это не critic в обычном понимании это для того чтобы обучать в памяти value_net то есть чтобы модель сохраняла инфу помагающую определять награду то есть была полесной и еще мне в идеале сделать чтобы градиент далше чем этоот слой dense не проходил именно по этому пути от dense val_net дальше нельзя 

2 сука я тебе скидывал эти логи смотреть надо внимательно а не хуйню нести 

3 







In [ ]:
1  это стандарт для ппо и при этом у меня 1 раз из 15 только

3 это тогже норма для ппо вроде да и не определяет огромный log или kl  а если и так то как испарвить